# SentinelMail — Ensemble: Weight Learning & Evaluation

Confidence-weighted **per-category late fusion** of the four models
(BERT, RoBERTa, Bi-LSTM, rule-based). This notebook fits per-label fusion
weights + thresholds on a **DEV** partition (the seeded 10% held-out slice of
`train` — the same slice neural checkpoint selection used) and reports all final
metrics on the **untouched TEST** split (the full ai4privacy `validation` split,
touched exactly once). DEV and TEST are fully disjoint, so the ensemble gets no
peek at the rows it is scored on (avoids tuning-on-test bias). Single models are
re-evaluated on the same TEST split for a fair, apples-to-apples RQ1 comparison.

**Prerequisite:** all four model checkpoints must exist under `models/*/checkpoint/`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

RESULTS_DIR = REPO_ROOT / "evaluation" / "results"
ENSEMBLE_DIR = REPO_ROOT / "ensemble"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
RANDOM_STATE = 42
THRESHOLD = 0.5
print(f"Device: {DEVICE}")

In [ ]:
from ensemble import (
    build_cache,
    load_probas,
    fit_weights,
    tune_thresholds,
    save_weights,
    save_thresholds,
)
from evaluation import (
    LABEL_COLS,
    compute_all_metrics,
    fuse,
    run_ablation,
    ensemble_vs_best_single,
    plot_ablation_table,
    plot_model_comparison_table,
    plot_confusion_matrices,
    plot_per_label_bars,
    plot_roc_curves,
    plot_pr_curves,
    save_results,
)

## 1. Build / Load Probability Caches (DEV + TEST)

`build_cache` runs each detector's `predict_proba` once over a split and caches
the arrays (BERT/RoBERTa CPU inference is the bottleneck). We build two disjoint
caches:

- **DEV** = `train_holdout` (seeded 10% slice of `train`) — used to fit weights/thresholds.
- **TEST** = `validation` (full ai4privacy validation split) — used for reporting only.

Both are idempotent — re-running is instant unless the split text changes. The
loader verifies an sha1 of the ordered emails so every model's rows are aligned.

In [ ]:
DEV_SPLIT = "train_holdout"   # seeded 10% held-out slice of `train`
TEST_SPLIT = "validation"     # untouched TEST split, reported once

build_cache(split=DEV_SPLIT, device=str(DEVICE))
build_cache(split=TEST_SPLIT, device=str(DEVICE))

dev_probas, dev_y, dev_texts = load_probas(split=DEV_SPLIT)
eval_probas, eval_y, eval_texts = load_probas(split=TEST_SPLIT)

print(f"DEV (fit):   {dev_y.shape[0]:,} emails")
print(f"TEST (report): {eval_y.shape[0]:,} emails; models: {list(eval_probas)}")
for name, arr in eval_probas.items():
    print(f"  {name:12s} {arr.shape}  range [{arr.min():.3f}, {arr.max():.3f}]")

## 2. DEV / TEST Partitions

DEV (`train_holdout`) and TEST (`validation`) come from disjoint dataset splits —
DEV is carved from `train`, TEST is the untouched ai4privacy `validation` split.
Weights and thresholds are fit on **DEV**; all reported metrics (ensemble and
single models) come from **TEST**. No in-notebook re-splitting of the reporting
split is needed.

In [ ]:
# DEV and TEST are already disjoint splits loaded above; no re-splitting needed.
assert dev_y.shape[0] > 0 and eval_y.shape[0] > 0
print(f"DEV: {dev_y.shape[0]:,} (train_holdout)   TEST: {eval_y.shape[0]:,} (validation)")

## 3. Fit Per-Category Weights (on DEV)

Per-label grid + Nelder-Mead search on the **true** binary F1. Fusion is
separable per label, so each label's four model-weights are optimised
independently. (`fit_weights_lbfgsb` is available as the spec-letter surrogate
alternative — see `ensemble/train_weights.py`.)

In [ ]:
weights = fit_weights(dev_probas, dev_y, threshold=THRESHOLD)
thresholds = tune_thresholds(dev_probas, dev_y, weights)

save_weights(weights, ENSEMBLE_DIR / "weights.json")
save_thresholds(thresholds, ENSEMBLE_DIR / "thresholds.json")
print("Saved weights.json and thresholds.json")
print("Tuned per-label thresholds:", thresholds)

### 3a. Learned Weights (RQ2 — does complementarity vary by category?)

In [ ]:
weights_df = pd.DataFrame(weights).T[LABEL_COLS].round(3)  # rows=model, cols=label
weights_df

## 4. Evaluate Ensemble (untouched TEST split — full `validation`)

In [ ]:
# Headline ensemble result uses the DEV-tuned per-label thresholds (comment 017, Decision A1).
y_pred_e, y_proba_e = fuse(eval_probas, weights, threshold=thresholds)
metrics = compute_all_metrics(eval_y, y_pred_e, y_proba=y_proba_e)

ci = metrics["macro_f1_ci"]
print(f"Ensemble macro-F1: {metrics['macro_f1']:.4f}  "
      f"95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
for label in LABEL_COLS:
    print(f"  {label:12s} F1={metrics['per_label'][label]['f1']:.4f}")

### 4a. Threshold Sensitivity (0.3 / 0.5 / 0.7 + tuned per-label)

In [ ]:
rows = []
for t in [0.3, 0.5, 0.7]:
    m = compute_all_metrics(eval_y, (y_proba_e >= t).astype(np.int32), n_bootstrap=50)
    row = {"config": f"fixed@{t}", "macro_f1": round(m["macro_f1"], 4)}
    for label in LABEL_COLS:
        row[f"{label}_f1"] = round(m["per_label"][label]["f1"], 4)
    rows.append(row)

# tuned per-label thresholds
y_pred_tuned = np.stack(
    [(y_proba_e[:, i] >= thresholds[label]).astype(np.int32)
     for i, label in enumerate(LABEL_COLS)], axis=1)
m = compute_all_metrics(eval_y, y_pred_tuned, n_bootstrap=50)
row = {"config": "tuned_per_label", "macro_f1": round(m["macro_f1"], 4)}
for label in LABEL_COLS:
    row[f"{label}_f1"] = round(m["per_label"][label]["f1"], 4)
rows.append(row)

pd.DataFrame(rows).set_index("config")

## 5. Ablation Study (RQ1 — does the ensemble beat any single model?)

`run_ablation` reports the full ensemble, each leave-one-model-out
configuration, every solo model, and the best single model — all on the untouched
TEST split (full `validation`, same rows as the ensemble) with 95% bootstrap CIs.

In [ ]:
# RQ1/ablation: the SAME DEV-tuned per-label thresholds are applied to the
# ensemble AND to every single model (comment 017 Decision B1 / comment 010),
# so all rows are apples-to-apples. NOTE: a single model's F1 in THIS table is
# computed at the tuned thresholds; that same model's own headline result file
# (evaluation/results/<model>_metrics.json) remains reported at 0.5.
ablation_df = run_ablation(eval_probas, weights, eval_y, threshold=thresholds)
plot_ablation_table(ablation_df)

# RQ1 significance gate: paired bootstrap on macro-F1 delta (ensemble - best single),
# same TEST rows resampled for both configs. Claim a gain only if the delta CI excludes 0.
rq1 = ensemble_vs_best_single(eval_probas, weights, eval_y, threshold=thresholds)
print(
    f"Ensemble macro-F1 = {rq1['ensemble_macro_f1']:.5f} vs "
    f"best single ({rq1['best_single_model']}) macro-F1 = {rq1['best_single_macro_f1']:.5f}"
)
print(
    f"delta = {rq1['mean_delta']:+.5f}  "
    f"95% CI [{rq1['ci_lower']:+.5f}, {rq1['ci_upper']:+.5f}]  "
    f"bootstrap p = {rq1['p_value']:.4f}"
)
if rq1['significant']:
    print('RQ1 VERDICT: SIGNIFICANT — delta CI excludes 0; ensemble outperforms the best single model.')
else:
    print('RQ1 VERDICT: NOT SIGNIFICANT — delta CI includes 0; no significant ensemble gain over the best single model.')


## 6. Model Comparison (recomputed on the same TEST split)

Single-model metrics are recomputed here from the cached TEST probabilities at
threshold 0.5 — the same untouched `validation` split the ensemble is reported
on, so the RQ1 comparison is apples-to-apples. The per-model result JSONs in
`evaluation/results/` should also be regenerated on this TEST split (see comment 001).

In [ ]:
results = {"ensemble": metrics}
for name, probas in eval_probas.items():
    y_pred_m = (probas >= THRESHOLD).astype(np.int32)
    # rule_based produces binary 0/1 flags, not calibrated probabilities -> y_proba=None
    y_proba_m = None if name == "rule_based" else probas
    results[name] = compute_all_metrics(eval_y, y_pred_m, y_proba=y_proba_m)

plot_model_comparison_table(results)

## 7. Confusion Matrices, Per-Label Bars, ROC & PR Curves

In [ ]:
_, fig = plot_confusion_matrices(eval_y, y_pred_e, normalize="true", return_fig=True)
fig

In [ ]:
_, fig = plot_per_label_bars(metrics, return_fig=True)
fig

In [ ]:
roc_data = plot_roc_curves(eval_y, y_proba_e)
pr_data = plot_pr_curves(eval_y, y_proba_e)

## 8. Save Results

In [ ]:
out_path = RESULTS_DIR / "ensemble_metrics.json"

save_results(
    metrics,
    path=out_path,
    model_name="ensemble_weighted_fusion",
    n_samples=eval_y.shape[0],
    threshold=thresholds,
    extra_metadata={
        "fusion": "per-category weighted late fusion",
        "optimizer": "per-label grid + Nelder-Mead on true F1",
        "dev_split": "train_holdout",
        "test_split": "validation",
        "random_state": RANDOM_STATE,
        "weights": weights,
        "thresholds": thresholds,
    },
)
print(f"Saved to {out_path}")